# 05 - Interpretabilita' e analisi errori

## Obiettivi didattici

1. Ispezionare i **coefficienti** della LogReg (modello interpretabile per costruzione) o le `feature_importances_` di RandomForest.
2. Studiare i **falsi negativi**: pazienti riammessi entro 30 giorni non intercettati dal modello.
3. Discutere i **limiti** del dataset (1999-2008, ICD-9, mancanza di dati socio-economici) e del modello.
4. (Opzionale) generare uno **SHAP summary** se la dipendenza extra e' installata.


In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

import joblib
from readmit_pipeline.config import MODELS_DIR, PipelineConfig
from readmit_pipeline.pipeline import prepare_data
from readmit_pipeline.interpretability import (
    feature_importance_or_coef, shap_summary,
)


In [ ]:
model = joblib.load(MODELS_DIR / 'best_model.joblib')
config = PipelineConfig(random_state=42)
_, X_test, _, y_test, _, _, _ = prepare_data(config)
# Naviga in TransformedTargetRegressor / Pipeline annidate.
inner = model
while hasattr(inner, 'named_steps') and 'feature_engineer' in inner.named_steps:
    # estrai pipeline modello (preprocessor + model) saltando il feature_engineer
    inner = type(inner)(steps=[(n, s) for n, s in inner.steps if n != 'feature_engineer'])
    break
top = feature_importance_or_coef(inner, top_n=20)
top


## Analisi falsi negativi


In [ ]:
y_score = model.predict_proba(X_test)[:, 1]
y_pred = (y_score >= 0.5).astype(int)
fn_mask = (y_test.values == 1) & (y_pred == 0)
print(f'Falsi negativi: {fn_mask.sum()} su {y_test.sum()} positivi reali ({fn_mask.sum()/y_test.sum()*100:.1f}%)')
X_test.loc[fn_mask].describe(include='all').T.head(15)


## Limiti e implicazioni

- **Temporali**: dati 1999-2008; le pratiche cliniche sono cambiate.
- **Coding**: ICD-9 non e' piu' lo standard (ICD-10 dal 2015 negli USA).
- **Mancanza di feature**: nessuna informazione su condizioni socio-economiche, supporto familiare, aderenza terapeutica post-dimissione.
- **Performance moderate**: AUC-ROC tipico in letteratura ~0.60-0.67. Il problema e' intrinsecamente difficile.
- **Etica**: includere `race` come feature predittiva e' un dilemma. La razza e' un costrutto sociale, non biologico; la sua inclusione puo' codificare disparita' strutturali piu' che fornire informazione.
